# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
df = pd.read_csv('AviationData.csv', encoding='latin1', low_memory=False)

print('Shape (rows, columns):', df.shape)
print()

df.head(3)

nan_counts = df.isnull().sum()
nan_pct    = (nan_counts / len(df) * 100).round(1)

nan_summary = pd.DataFrame({'NaN Count': nan_counts, 'NaN %': nan_pct})

print('Missing value summary (columns with any NaNs):')
nan_summary[nan_summary['NaN Count'] > 0].sort_values('NaN %', ascending=False)

df.describe()

Shape (rows, columns): (88889, 31)

Missing value summary (columns with any NaNs):


,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
df = pd.read_csv('AviationData.csv', encoding='latin1', low_memory=False)

print('Shape (rows, columns):', df.shape)
print()

df.head(3)

nan_counts = df.isnull().sum()
nan_pct    = (nan_counts / len(df) * 100).round(1)

nan_summary = pd.DataFrame({'NaN Count': nan_counts, 'NaN %': nan_pct})

print('Missing value summary (columns with any NaNs):')
nan_summary[nan_summary['NaN Count'] > 0].sort_values('NaN %', ascending=False)

df.describe()

Shape (rows, columns): (88889, 31)

Missing value summary (columns with any NaNs):


,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries',
               'Total.Minor.Injuries', 'Total.Uninjured']

print('Injury column NaN counts:')
print(df[injury_cols].isnull().sum())
print()
print('Injury column statistics:')
df[injury_cols].describe()

df[injury_cols] = df[injury_cols].fillna(0)


df['Total.Aboard'] = (df['Total.Fatal.Injuries']
                    + df['Total.Serious.Injuries']
                    + df['Total.Minor.Injuries']
                    + df['Total.Uninjured'])

df['Fatal.Serious.Rate'] = (
    (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries'])
    / df['Total.Aboard'].clip(lower=1)
)

print('Fatal.Serious.Rate statistics:')
print(df['Fatal.Serious.Rate'].describe())

Injury column NaN counts:
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
dtype: int64

Injury column statistics:
Fatal.Serious.Rate statistics:
count    88889.000000
mean         0.285751
std          0.435478
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000
Name: Fatal.Serious.Rate, dtype: float64


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
print('Aircraft.damage value counts:')
print(df['Aircraft.damage'].value_counts())
print('NaN count:', df['Aircraft.damage'].isna().sum())

df['Aircraft.damage'] = df['Aircraft.damage'].str.strip().str.title()

def classify_destroyed(val):
    if val == 'Destroyed':
        return 1
    elif val in ('Substantial', 'Minor'):
        return 0
    else:  # 'Unknown' or NaN
        return np.nan

df['Is.Destroyed'] = df['Aircraft.damage'].apply(classify_destroyed)

print('Is.Destroyed value counts:')
print(df['Is.Destroyed'].value_counts())
print('NaN count:', df['Is.Destroyed'].isna().sum())

Aircraft.damage value counts:
Aircraft.damage
Substantial    64148
Destroyed      18623
Minor           2805
Unknown          119
Name: count, dtype: int64
NaN count: 3194
Is.Destroyed value counts:
Is.Destroyed
0.0    66953
1.0    18623
Name: count, dtype: int64
NaN count: 3313


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [6]:
print('Top 20 raw Make values:')
print(df['Make'].value_counts().head(20))
print()
print('Total unique Makes (raw):', df['Make'].nunique())
print('NaN count:', df['Make'].isna().sum())


df['Make'] = df['Make'].str.strip().str.title()

df = df.dropna(subset=['Make'])
print('Rows after dropping NaN Makes:', len(df))


make_counts = df['Make'].value_counts()
valid_makes  = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)].copy()

print('Rows after removing rare Makes:', len(df))
print('Unique Makes remaining:', df['Make'].nunique())
print()
print('Final Make counts:')
print(df['Make'].value_counts())

Top 20 raw Make values:
Make
Cessna               22227
Piper                12029
CESSNA                4922
Beech                 4330
PIPER                 2841
Bell                  2134
Boeing                1594
BOEING                1151
Grumman               1094
Mooney                1092
BEECH                 1042
Robinson               946
Bellanca               886
Hughes                 795
Schweizer              629
Air Tractor            595
BELL                   588
Mcdonnell Douglas      526
Aeronca                487
Maule                  445
Name: count, dtype: int64

Total unique Makes (raw): 8237
NaN count: 63
Rows after dropping NaN Makes: 88826
Rows after removing rare Makes: 74650
Unique Makes remaining: 98

Final Make counts:
Make
Cessna                         27149
Piper                          14870
Beech                           5372
Boeing                          2745
Bell                            2722
                               ...  
Lancair   

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
print('Model NaN count:', df['Model'].isna().sum())

df = df.dropna(subset=['Model'])
print('Rows after dropping NaN Models:', len(df))

df['Model'] = df['Model'].str.strip().str.upper()

model_make_pairs = df.groupby('Model')['Make'].nunique()
shared_models    = model_make_pairs[model_make_pairs > 1]

print(f'\nModel names that appear under multiple Makes: {len(shared_models)}')
print('Some examples:')
print(shared_models.head(10))

df['Make.Model'] = df['Make'] + ' ' + df['Model']

print('Most common Make.Model combinations:')
print(df['Make.Model'].value_counts().head(10))

Model NaN count: 23
Rows after dropping NaN Models: 74627

Model names that appear under multiple Makes: 495
Some examples:
Model
100        4
100-180    2
105        2
109        2
109A       2
112        5
112A       3
112TC      3
112TCA     3
114        2
Name: Make, dtype: int64
Most common Make.Model combinations:
Make.Model
Cessna 152         2366
Cessna 172         1753
Cessna 172N        1163
Piper PA-28-140     932
Cessna 150          829
Cessna 172M         798
Cessna 172P         689
Cessna 182          659
Cessna 180          621
Cessna 150M         585
Name: count, dtype: int64


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
print('=== Engine.Type (raw) ===')
print(df['Engine.Type'].value_counts())
print('NaN count:', df['Engine.Type'].isna().sum())
print()

df['Engine.Type'] = df['Engine.Type'].str.strip().str.title()
df['Engine.Type'] = df['Engine.Type'].replace({'Unknown': np.nan, 'Unk': np.nan, 'None': np.nan})

print('Engine.Type after cleaning:')
print(df['Engine.Type'].value_counts())

=== Engine.Type (raw) ===
Engine.Type
Reciprocating      58626
Turbo Shaft         3116
Turbo Prop          2886
Turbo Fan           2220
Unknown             1611
Turbo Jet            565
Geared Turbofan       12
LR                     1
UNK                    1
NONE                   1
Name: count, dtype: int64
NaN count: 5588

Engine.Type after cleaning:
Engine.Type
Reciprocating      58626
Turbo Shaft         3116
Turbo Prop          2886
Turbo Fan           2220
Turbo Jet            565
Geared Turbofan       12
Lr                     1
Name: count, dtype: int64


In [9]:
print('=== Weather.Condition (raw) ===')
print(df['Weather.Condition'].value_counts())
print('NaN count:', df['Weather.Condition'].isna().sum())
print()

df['Weather.Condition'] = df['Weather.Condition'].str.strip()
df['Weather.Condition'] = df['Weather.Condition'].replace({'UNK': np.nan, 'Unk': np.nan})

print('Weather.Condition after cleaning:')
print(df['Weather.Condition'].value_counts())

=== Weather.Condition (raw) ===
Weather.Condition
VMC    64185
IMC     5601
UNK      802
Unk      231
Name: count, dtype: int64
NaN count: 3808

Weather.Condition after cleaning:
Weather.Condition
VMC    64185
IMC     5601
Name: count, dtype: int64


In [10]:
print('=== Number.of.Engines (raw) ===')
print(df['Number.of.Engines'].value_counts())
print('NaN count:', df['Number.of.Engines'].isna().sum())
print()

df['Number.of.Engines'] = df['Number.of.Engines'].replace(0, np.nan)

print('Number.of.Engines after cleaning:')
print(df['Number.of.Engines'].value_counts())

=== Number.of.Engines (raw) ===
Number.of.Engines
1.0    57842
2.0    10121
0.0      795
3.0      450
4.0      411
8.0        1
Name: count, dtype: int64
NaN count: 5007

Number.of.Engines after cleaning:
Number.of.Engines
1.0    57842
2.0    10121
3.0      450
4.0      411
8.0        1
Name: count, dtype: int64


In [11]:
print('=== Purpose.of.flight (raw) ===')
print(df['Purpose.of.flight'].value_counts())
print('NaN count:', df['Purpose.of.flight'].isna().sum())
print()


df['Purpose.of.flight'] = df['Purpose.of.flight'].str.strip().str.title()
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({'Unknown': np.nan})

print('Purpose.of.flight after cleaning:')
print(df['Purpose.of.flight'].value_counts())

=== Purpose.of.flight (raw) ===
Purpose.of.flight
Personal                     39217
Instructional                 9845
Unknown                       6129
Aerial Application            4293
Business                      3753
Positioning                   1451
Other Work Use                1069
Aerial Observation             727
Ferry                          708
Public Aircraft                666
Executive/corporate            478
Skydiving                      177
Flight Test                    165
Banner Tow                      97
External Load                   85
Public Aircraft - Federal       76
Public Aircraft - State         58
Public Aircraft - Local         52
Glider Tow                      41
Firefighting                    33
Air Race show                   30
Air Race/show                   18
Air Drop                        10
PUBS                             3
ASHO                             3
PUBL                             1
Name: count, dtype: int64
NaN count: 544

In [12]:
print('=== Broad.phase.of.flight (raw) ===')
print(df['Broad.phase.of.flight'].value_counts())
print('NaN count:', df['Broad.phase.of.flight'].isna().sum())
print()


df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.strip().str.title()
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace({'Unknown': np.nan})

print('Broad.phase.of.flight after cleaning:')
print(df['Broad.phase.of.flight'].value_counts())

=== Broad.phase.of.flight (raw) ===
Broad.phase.of.flight
Landing        13979
Takeoff        10550
Cruise          9044
Maneuvering     6713
Approach        5602
Taxi            1809
Climb           1752
Descent         1666
Go-around       1251
Standing         865
Unknown          454
Other             91
Name: count, dtype: int64
NaN count: 20851

Broad.phase.of.flight after cleaning:
Broad.phase.of.flight
Landing        13979
Takeoff        10550
Cruise          9044
Maneuvering     6713
Approach        5602
Taxi            1809
Climb           1752
Descent         1666
Go-Around       1251
Standing         865
Other             91
Name: count, dtype: int64


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [13]:
# Re-calculate NaN percentages after cleaning
nan_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

print('NaN percentage per column (after cleaning):')
print(nan_pct.round(1))

NaN percentage per column (after cleaning):
Schedule                  84.7
Air.carrier               81.8
FAR.Description           66.6
Aircraft.Category         66.3
Longitude                 64.7
Latitude                  64.7
Airport.Code              44.0
Airport.Name              40.9
Broad.phase.of.flight     28.5
Publication.Date          16.5
Purpose.of.flight         15.5
Engine.Type                9.6
Number.of.Engines          7.8
Report.Status              6.9
Weather.Condition          6.5
Is.Destroyed               3.8
Aircraft.damage            3.7
Registration.Number        1.6
Injury.Severity            1.2
Country                    0.3
Amateur.Built              0.1
Location                   0.1
Total.Aboard               0.0
Total.Uninjured            0.0
Fatal.Serious.Rate         0.0
Event.Id                   0.0
Total.Minor.Injuries       0.0
Total.Serious.Injuries     0.0
Total.Fatal.Injuries       0.0
Investigation.Type         0.0
Model                     

In [14]:
# Drop columns where more than 50% of rows are missing.
threshold = 50.0  # percent

cols_to_drop = nan_pct[nan_pct > threshold].index.tolist()
print(f'Columns to drop (>{threshold}% NaN):')
for c in cols_to_drop:
    print(f'  {c}: {nan_pct[c]:.1f}% missing')

df = df.drop(columns=cols_to_drop)
print()
print('Remaining columns:', df.shape[1])
print('Remaining rows:   ', df.shape[0])
print()
print('Columns kept:', df.columns.tolist())

Columns to drop (>50.0% NaN):
  Schedule: 84.7% missing
  Air.carrier: 81.8% missing
  FAR.Description: 66.6% missing
  Aircraft.Category: 66.3% missing
  Longitude: 64.7% missing
  Latitude: 64.7% missing



Remaining columns: 29
Remaining rows:    74627

Columns kept: ['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'Purpose.of.flight', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date', 'Total.Aboard', 'Fatal.Serious.Rate', 'Is.Destroyed', 'Make.Model']


In [15]:
# Preview the cleaned dataframe
print('Final dataframe shape:', df.shape)
df.head()

Final dataframe shape: (74627, 29)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Airport.Code,Airport.Name,Injury.Severity,Aircraft.damage,...,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.Aboard,Fatal.Serious.Rate,Is.Destroyed,Make.Model
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,Fatal(2),Destroyed,...,0.0,0.0,NaN,Cruise,Probable Cause,NaN,2.0,1.0,1.0,Stinson 108-3
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,Fatal(4),Destroyed,...,0.0,0.0,NaN,NaN,Probable Cause,19-09-1996,4.0,1.0,1.0,Piper PA24-180
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,NaN,NaN,Fatal(3),Destroyed,...,0.0,0.0,IMC,Cruise,Probable Cause,26-02-2007,3.0,1.0,1.0,Cessna 172M
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,Fatal(2),Destroyed,...,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000,2.0,1.0,1.0,Rockwell 112
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,Fatal(1),Destroyed,...,0.0,0.0,VMC,Approach,Probable Cause,16-04-1980,3.0,1.0,1.0,Cessna 501


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [16]:
# Save the cleaned dataframe to a new CSV file.
df.to_csv('AviationData_Cleaned.csv', index=False)

print('Cleaned data saved to: AviationData_Cleaned.csv')
print('Final shape:', df.shape)
print()
print('Summary of key derived columns added:')
print('  Total.Aboard      -- estimated number of people on board per accident')
print('  Fatal.Serious.Rate -- fraction of people aboard with fatal/serious injuries (0=safest, 1=worst)')
print('  Is.Destroyed       -- 1 if aircraft was destroyed, 0 if not, NaN if unknown')
print('  Make.Model         -- unique identifier combining Make + Model (e.g. Cessna 172)')
print('  Year               -- year extracted from Event.Date')

Cleaned data saved to: AviationData_Cleaned.csv
Final shape: (74627, 29)

Summary of key derived columns added:
  Total.Aboard      -- estimated number of people on board per accident
  Fatal.Serious.Rate -- fraction of people aboard with fatal/serious injuries (0=safest, 1=worst)
  Is.Destroyed       -- 1 if aircraft was destroyed, 0 if not, NaN if unknown
  Make.Model         -- unique identifier combining Make + Model (e.g. Cessna 172)
  Year               -- year extracted from Event.Date
